# Implementation

In [16]:
# Importing the data

from pathlib import Path

data_path = Path("../data/plants.data") # path to plant data

with open(data_path, "r", encoding="iso-8859-1", errors="replace") as f: # import the plant data
    data = [line.strip() for line in f if line.strip()]

for plant in data[0:5]:
    print(plant)

abelia,fl,nc
abelia x grandiflora,fl,nc
abelmoschus,ct,dc,fl,hi,il,ky,la,md,mi,ms,nc,sc,va,pr,vi
abelmoschus esculentus,ct,dc,fl,il,ky,la,md,mi,ms,nc,sc,va,pr,vi
abelmoschus moschatus,hi,pr


In [17]:
print("Data type:", type(data))
print("Data instances:", len(data))

Data type: <class 'list'>
Data instances: 34781


In [18]:
# Data cleaning

import pandas as pd

plant_names = [] # list of plant names
state_lists = [] # list of states where each plant grows

for plant in data:
    parts = [part.strip() for part in plant.split(",")] # delimit by ","
    name = parts[0] # plant name is the first part
    states = parts[1:] # remaining parts are the states where the plant grows
    plant_names.append(name) # append to list of all plant names
    state_lists.append(states) # append to list of where each plant grows

for name in plant_names[:5]:
    print(name)

print("-"*25)

for states in state_lists[:5]:
    print(states)

abelia
abelia x grandiflora
abelmoschus
abelmoschus esculentus
abelmoschus moschatus
-------------------------
['fl', 'nc']
['fl', 'nc']
['ct', 'dc', 'fl', 'hi', 'il', 'ky', 'la', 'md', 'mi', 'ms', 'nc', 'sc', 'va', 'pr', 'vi']
['ct', 'dc', 'fl', 'il', 'ky', 'la', 'md', 'mi', 'ms', 'nc', 'sc', 'va', 'pr', 'vi']
['hi', 'pr']


In [19]:
# Data frame construction from scratch

unique_states = sorted({st for states in state_lists for st in states}) # build list of unique states/provinces

print("-"*25)
print("Total plants:", len(plant_names))
print("Total states/provinces:", len(unique_states))
print("-"*25)

# Create the binary matrix

import numpy as np

# rows = plants, columns = states
state_to_col = {st: i for i, st in enumerate(unique_states)} # state to column mapping
binary_matrix = np.zeros((len(plant_names), len(unique_states)), dtype=int) # matrix of 0s

for row, states in enumerate(state_lists): # iterates through each list of states where a plant grows
    for st in states: # iterates through each state
        col = state_to_col[st] # get column mapping
        binary_matrix[row, col] = 1 # changes value to a 1 if the given plant grows in the given state

df = pd.DataFrame(binary_matrix, index=plant_names, columns=unique_states) # create data frame
print(df.head(5))

-------------------------
Total plants: 34781
Total states/provinces: 70
-------------------------
                        ab  ak  al  ar  az  bc  ca  co  ct  dc  ...  tx  ut  \
abelia                   0   0   0   0   0   0   0   0   0   0  ...   0   0   
abelia x grandiflora     0   0   0   0   0   0   0   0   0   0  ...   0   0   
abelmoschus              0   0   0   0   0   0   0   0   1   1  ...   0   0   
abelmoschus esculentus   0   0   0   0   0   0   0   0   1   1  ...   0   0   
abelmoschus moschatus    0   0   0   0   0   0   0   0   0   0  ...   0   0   

                        va  vi  vt  wa  wi  wv  wy  yt  
abelia                   0   0   0   0   0   0   0   0  
abelia x grandiflora     0   0   0   0   0   0   0   0  
abelmoschus              1   1   0   0   0   0   0   0  
abelmoschus esculentus   1   1   0   0   0   0   0   0  
abelmoschus moschatus    0   0   0   0   0   0   0   0  

[5 rows x 70 columns]


In [20]:
# Data frame construction with scikit-learn

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer() # creates binary matrix
binary_matrix_sk = mlb.fit_transform(state_lists) # each state becomes a column, each row has a 1/0 reflecting if the plant grows there

df_sk = pd.DataFrame(binary_matrix_sk, index=plant_names, columns=mlb.classes_) # create data frame

print("-"*25)
print("Total plants:", binary_matrix_sk.shape[0])
print("Total states/provinces:", binary_matrix_sk.shape[1])
print("-"*25)

print(df_sk.head(5)) # first 5 rows/plants

-------------------------
Total plants: 34781
Total states/provinces: 70
-------------------------
                        ab  ak  al  ar  az  bc  ca  co  ct  dc  ...  tx  ut  \
abelia                   0   0   0   0   0   0   0   0   0   0  ...   0   0   
abelia x grandiflora     0   0   0   0   0   0   0   0   0   0  ...   0   0   
abelmoschus              0   0   0   0   0   0   0   0   1   1  ...   0   0   
abelmoschus esculentus   0   0   0   0   0   0   0   0   1   1  ...   0   0   
abelmoschus moschatus    0   0   0   0   0   0   0   0   0   0  ...   0   0   

                        va  vi  vt  wa  wi  wv  wy  yt  
abelia                   0   0   0   0   0   0   0   0  
abelia x grandiflora     0   0   0   0   0   0   0   0  
abelmoschus              1   1   0   0   0   0   0   0  
abelmoschus esculentus   1   1   0   0   0   0   0   0  
abelmoschus moschatus    0   0   0   0   0   0   0   0  

[5 rows x 70 columns]


In [21]:
print("Scratch implementation equivalent to scikit-learn?", df.equals(df_sk))

Scratch implementation equivalent to scikit-learn? True


In [22]:
# Hierarchical clustering implementation from scratch

def jaccard_distance(x1, x2):

    inter = np.logical_and(x1 == 1, x2 == 1).sum() # number of states where both plants grow
    union = np.logical_or(x1 == 1, x2 == 1).sum() # number of states where either plant grows

    if union == 0: # if neither plant grows anywhere
        return 0.0
    
    return 1.0 - inter / union # return jaccard distance

def distance_matrix(X):

    n = X.shape[0] # number of plants
    dm = np.zeros((n, n), dtype=float) # matrix to store distance between every pair of plants
    
    for i in range(n): # iterate through each plant
        for j in range(i + 1, n): # iterate through other plants that have not been compared to the given plant yet
            d = jaccard_distance(X[i], X[j]) # jaccard distance
            dm[i, j] = dm[j, i] = d # distance is symmetric, so fill in both
    
    return dm

def agglomerative(X):

    n = X.shape[0] # number of plants
    dm = distance_matrix(X) # create the distance matrix

    clusters = set(range(n)) # start with each plant as its own cluster
    cluster_size = {i: 1 for i in range(n)} # initial cluster sizes are all 1
    next_id = n # id for the next new cluster from the first merge
    merges = [] # each item will have: [cluster 1 id, cluster 2 id, distance, new cluster size]

    dist = {} # distances between current clusters
    for i in range(n):
        for j in range(i + 1, n):
            dist[(i, j)] = dm[i, j] # creates dictionary of distances

    while len(clusters) > 1:

        merge_pair = None # initiate merge pair to none
        merge_dist = float("inf") # initiate distance of merge pair to inf

        for (i, j), d in dist.items(): # iterate through cluster distances
            if i in clusters and j in clusters and d < merge_dist: # find closest cluster
                merge_dist = d
                merge_pair = (i, j)
        
        a, b = merge_pair # merge clusters a and b
        new_id = next_id # id for newly merged cluster
        next_id += 1

        new_size = cluster_size[a] + cluster_size[b] # size of newly merged cluster
        merges.append([a, b, merge_dist, new_size]) # store list of merges
        clusters.remove(a) # remove merged cluster a
        clusters.remove(b) # remove merged cluster b

        for c in list(clusters): # update distances from newly merged cluster to other clusters

            key_ac = (min(a, c), max(a, c)) # dictionary key for distance between cluster a and c
            key_bc = (min(b, c), max(b, c)) # dictionary key for distance between cluster b and c
            # Note: (ac, ca should be the same) and (bc, cb should be the same)
            d_ac = dist[key_ac] # stored distance between clusters a and c
            d_bc = dist[key_bc] # stored distance between clusters b and c

            # jaccard distance from merged cluster to cluster c
            d_newc = (cluster_size[a] * d_ac + cluster_size[b] * d_bc) / new_size
            # stores new distance between merged cluster and cluster c
            dist[(min(new_id, c), max(new_id, c))] = d_newc

        clusters.add(new_id) # add newly merged cluster ab
        cluster_size[new_id] = new_size # add size of newly merged cluster ab

        if len(merges) % 100 == 0: # for tracking clustering progress
            print( 
                f"[merge {len(merges)}] "
                f"merge ({a}, {b}) at distance={merge_dist:.4f} -> size={new_size}, "
                f"active clusters after merge={len(clusters)}"
            )

    return np.array(merges, dtype=float)

In [ ]:
# only using a subset of the dataset due to computational demand of scratch implementation

import random

k = min(5000, len(plant_names)) # only use a maximum of 5000 samples for clustering
rng = random.Random(42)  # fixed seed for reproducibility
indices = rng.sample(range(len(plant_names)), k) # randomly chosen samples to use for clustering

plant_names_sub = [plant_names[i] for i in indices] # randomly selected plant names
state_lists_sub = [state_lists[i] for i in indices] # corresponding state lists for randomly selected plants

mlb_sub = MultiLabelBinarizer()
binary_matrix_sub = mlb_sub.fit_transform(state_lists_sub) # new binary matrix for subset of the data
clustering_results = agglomerative(binary_matrix_sub)

print("-"*25)
print(clustering_results.shape)

[merge 100] merge (232, 262) at distance=0.0000 -> size=2, active clusters after merge=4900
[merge 200] merge (494, 502) at distance=0.0000 -> size=2, active clusters after merge=4800
[merge 300] merge (774, 796) at distance=0.0000 -> size=2, active clusters after merge=4700
[merge 400] merge (1072, 1088) at distance=0.0000 -> size=2, active clusters after merge=4600
[merge 500] merge (1341, 1474) at distance=0.0000 -> size=2, active clusters after merge=4500
[merge 600] merge (1645, 1752) at distance=0.0000 -> size=2, active clusters after merge=4400
[merge 700] merge (1954, 4296) at distance=0.0000 -> size=2, active clusters after merge=4300
[merge 800] merge (2321, 3390) at distance=0.0000 -> size=2, active clusters after merge=4200
[merge 900] merge (2637, 2640) at distance=0.0000 -> size=2, active clusters after merge=4100


In [ ]:
# Hierarchical clustering implementation with scipy

from scipy.cluster.hierarchy import linkage

# uses jaccard distance and average link as implemented previously from scratch
clustering_results_sp = linkage(binary_matrix_sk, method="average", metric="jaccard")

print(clustering_results_sp.shape)

(34780, 4)


In [ ]:
# Cluster sizes

from scipy.cluster.hierarchy import fcluster

Z = clustering_results_sp # abbreviated variable name
k = 11 # fixed number of clusters, trial & error to find cluster number with 5 clusters >1000 plant species
labels = fcluster(Z, t=k, criterion="maxclust") # split clustering into k flat clusters
cluster_sizes = pd.Series(labels).value_counts().sort_index() # number of plant species in each cluster

print("-"*25)
print("Cluster sizes")
for cluster_id, size in cluster_sizes.items():
    print(f"Cluster {cluster_id}: {size} species")

-------------------------
Cluster sizes
Cluster 1: 10 species
Cluster 2: 3585 species
Cluster 3: 1991 species
Cluster 4: 8 species
Cluster 5: 1521 species
Cluster 6: 17032 species
Cluster 7: 262 species
Cluster 8: 51 species
Cluster 9: 10197 species
Cluster 10: 15 species
Cluster 11: 109 species


In [ ]:
# Species in the 5 clusters with >1000 species

top_cluster_ids = [2, 3, 5, 6, 9] # clusters with >1000 plant species

cluster_species_dict = {} # store species list by cluster

for cluster_id in top_cluster_ids: # for each of the top clusters
    species_in_cluster = [ # list of plant species in the current cluster
        plant_names[i]
        for i, label in enumerate(labels)
        if label == cluster_id
    ]
    # add species list as dataframe column
    cluster_species_dict[f"Cluster {cluster_id}"] = pd.Series(species_in_cluster)

top_clusters_df = pd.DataFrame(cluster_species_dict) # create dataframe (each col = 1 cluster)

output_path = "top_cluster_species.csv" # output csv path
top_clusters_df.to_csv(output_path, index=False) # export dataframe to csv for analysis

print("-" * 25)
print(f"Top cluster species exported to: {output_path}")

-------------------------
Top cluster species exported to: top_cluster_species.csv


In [ ]:
# Count region representation within each top cluster

cluster_df = pd.DataFrame({ # dict of plant species to cluster id
    "species": plant_names,
    "cluster_id": labels
})

print("-" * 25)
print("Region counts by top cluster")

for cluster_id in top_cluster_ids: # for each of the top clusters

    # use previous data frame but drop na values
    species_in_cluster = cluster_species_dict[f"Cluster {cluster_id}"].dropna()

    # use binary matrix to count region representation
    cluster_region_counts = df_sk.loc[species_in_cluster].sum(axis=0)

    # keep only regions represented by at least one species in this cluster
    cluster_region_counts = cluster_region_counts[
        cluster_region_counts > 0
    ].sort_values(ascending=False)

    # number of regions represented by species in the cluster
    num_regions_represented = len(cluster_region_counts)

    print("-" * 25)
    print(f"Cluster {cluster_id}: {len(species_in_cluster)} species")
    print(f"Regions represented: {num_regions_represented}")
    print("Regions represented:")

    for region, count in cluster_region_counts.items():
        print(f"{region}: {int(count)}")

-------------------------
Region counts by top cluster
-------------------------
Cluster 2: 3585 species
Regions represented: 30
Regions represented:
pr: 3483
vi: 1452
fl: 1010
hi: 473
tx: 162
la: 111
al: 50
ga: 43
ca: 29
az: 26
md: 19
sc: 19
ms: 15
nc: 14
ar: 10
dc: 5
nj: 5
nm: 5
ma: 3
va: 3
or: 3
ny: 3
ok: 2
ct: 2
me: 1
ky: 1
il: 1
ut: 1
wi: 1
wv: 1
-------------------------
Cluster 3: 1991 species
Regions represented: 21
Regions represented:
hi: 1987
fl: 57
vi: 9
ca: 8
la: 8
md: 8
al: 7
ri: 5
tx: 5
nc: 4
az: 3
or: 3
pa: 3
sc: 3
va: 3
nj: 2
ga: 2
wi: 2
pr: 1
mo: 1
ct: 1
-------------------------
Cluster 5: 1521 species
Regions represented: 67
Regions represented:
ak: 938
yt: 631
nt: 576
bc: 572
qc: 522
on: 382
nu: 375
ab: 363
gl: 320
lb: 283
mb: 267
nf: 252
sk: 171
mt: 131
wa: 120
ns: 114
nb: 100
me: 83
or: 82
wy: 82
co: 69
id: 66
mi: 60
fraspm: 59
ny: 54
ca: 54
nh: 49
wi: 38
mn: 35
ma: 32
dengl: 32
vt: 29
pe: 28
ut: 20
pa: 18
ct: 12
sd: 9
oh: 8
ia: 7
nm: 7
ri: 7
nv: 7
va: 7
nd: 6
tn

In [ ]:
# Species of interest

cluster_df = pd.DataFrame({ # dataframe mapping plant species to cluster IDs
    "species": plant_names,
    "cluster_id": labels
})

species_to_find = [
    "allium yosemitense",
    "sabal palmetto",
    "cycas revoluta",
    "rosa gallica",
    "lagerstroemia",
    "syringa",
    "hibiscus syriacus",
    "huperzia lucidula",
    "huperzia nutans"
]

print("-" * 25)
print("Cluster assignments for selected species")

for species_query in species_to_find: # find the cluster each species belongs to
    matches = cluster_df[
        cluster_df["species"].str.lower().str.contains(
            species_query.lower(),
            na=False
        )
    ]

    first_match = matches.iloc[0]

    print(f"{first_match['species']}: Cluster {first_match['cluster_id']}")

-------------------------
Cluster assignments for selected species
allium yosemitense: Cluster 6
sabal palmetto: Cluster 9
cycas revoluta: Cluster 2
rosa gallica: Cluster 9
lagerstroemia: Cluster 9
syringa: Cluster 9
hibiscus syriacus: Cluster 9
huperzia lucidula: Cluster 9
huperzia nutans: Cluster 3


### The full written discussion is available in reports/project_report.md